In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd

BASE_DIR = "drive/MyDrive/CSML/data"
OUT_DIR = os.path.join(BASE_DIR, "final_cross_domain_results")

raw = pd.read_csv(os.path.join(OUT_DIR, "final_cross_domain_raw.csv"))

# ----------------------------------------------------------------
# The definitive collapse check: for each trained model, look at
# its performance on its OWN source domain's test set (source==target).
# A model that collapsed to majority-class prediction will show
# test_pred_real_frac pinned near 0.0 or 1.0 on ITS OWN test set,
# regardless of what happened transiently mid-training (which is
# all epoch_first_collapse captured -- a transient flag, not a
# final-state flag).
# ----------------------------------------------------------------
in_domain = raw[raw["source"] == raw["target"]].copy()
in_domain["collapsed_final"] = (
    (in_domain["test_pred_real_frac"] >= 0.98) | (in_domain["test_pred_real_frac"] <= 0.02)
)

print("=== In-domain (source==target) results per run ===")
print(in_domain[[
    "model_type", "source", "seed", "test_macro_f1", "test_acc",
    "test_pred_real_frac", "epoch_first_collapse", "collapsed_final"
]].to_string(index=False))

n_collapsed_final = in_domain["collapsed_final"].sum()
n_total = len(in_domain)
print(f"\n=== Verification result ===")
print(f"Runs with FINAL-state collapse (in-domain pred_real_frac >= 0.98 or <= 0.02): "
      f"{n_collapsed_final} / {n_total}")

n_transient_flag = (in_domain["epoch_first_collapse"] >= 0).sum()
print(f"Runs with a TRANSIENT collapse flag at some epoch (informational only): "
      f"{n_transient_flag} / {n_total}")
print("(A transient flag with collapsed_final=False means the model recovered "
      "before the best-val-F1 checkpoint was selected -- expected given class weighting.)")

# Also flag any known collapse-fingerprint F1 values, as an independent cross-check
KNOWN_COLLAPSE_F1 = {
    ("coaid", "coaid"): [0.144201, 0.454000],
    ("fakenewsnet", "fakenewsnet"): [0.198618, 0.429274],
}
print("\n=== Cross-check against known collapse-fingerprint F1 values ===")
for (src, tgt), bad_values in KNOWN_COLLAPSE_F1.items():
    subset = in_domain[(in_domain["source"] == src) & (in_domain["target"] == tgt)]
    for _, row in subset.iterrows():
        matches_fingerprint = any(abs(row["test_macro_f1"] - v) < 1e-4 for v in bad_values)
        print(
            f"  {row['model_type']:6s} {src:12s} seed={row['seed']} "
            f"test_F1={row['test_macro_f1']:.4f}  matches_known_collapse_value={matches_fingerprint}"
        )

print(f"\nSaved nothing new -- this is a read-only verification against: {os.path.join(OUT_DIR, 'final_cross_domain_raw.csv')}")
